In [ ]:
from typing import TypedDict
from pydantic import BaseModel, ValidationError

### What TypedDict is
`TypedDict` (from typing) is just a type annotation for a regular dict. At runtime, an instance is a plain dict — there is no special class, no methods, no overhead.

In [ ]:
class Point(TypedDict):
    x: int
    y: int


p: Point = {"x": 1, "y": 2}  # literally a dict
type(p)

The annotations exist only for static type checkers (mypy, Pyright). At runtime, nothing is enforced:

In [ ]:
p: Point = {"x": "not an int", "z": 99}

### What Pydantic is
A Pydantic `BaseModel` is a real class with a metaclass that validates and coerces data at construction time and stores fields as attributes.

In [ ]:
class Point(BaseModel):
    x: int
    y: int


p = Point(x="1", y=2)  # "1" -> coerced to int 1
p.x  # attribute access, not p["x"]

In [ ]:
try:
    Point(x="abc", y=2)
except ValidationError as e:
    print("Incorrect type provided", e)

### The core differences

| Aspect | `TypedDict` | Pydantic `BaseModel` |
|---|---|---|
| Runtime type | plain `dict` | custom class instance |
| Validation | none (static only) | full, at construction |
| Type coercion | none | yes (`"1"` -> `1`, etc.) |
| Access | `obj["x"]` | `obj.x` |
| Memory per instance | low (just a dict) | higher (object + field machinery) |
| Construction speed | instant (dict literal) | slower (validation runs) |
| Serialization | manual | built-in (`.model_dump()`, JSON) |
| Default values | not really | yes, with `Field(...)` |
| Methods/validators | no | yes (`@field_validator`, etc.) |

`TypedDict` has no concept of defaults; every required key must be present (mark keys optional with `NotRequired`).

### When to use which

**Use `TypedDict` when:**
- The data is already trusted/internal (you produced it in code).
- You want type-checker hints without runtime cost.
- You need plain `dict` semantics (e.g. LangGraph state, which merges/updates dict keys).
- Hot paths where construction count is huge.

**Use Pydantic when:**
- Data crosses a **trust boundary**: API requests, LLM outputs, config files, user input. You want it validated/rejected early.
- You need coercion, defaults, custom validators, or serialization to/from JSON.

### In project specifically

- `RAGState` as `TypedDict` → internal graph state, needs dict-merge semantics, no validation overhead on every node transition.
- `response_format` models + `Config` as Pydantic → validating LLM output and env/TOML config, both untrusted external sources where a malformed value should fail loudly.

A common hybrid pattern: keep graph state as `TypedDict`, but make individual values inside it Pydantic models when those values come from the LLM (e.g. the intent-classification result). That way you get cheap state plumbing plus validated boundaries where it counts.

### The hybrid pattern: example

The graph state is a `TypedDict` (cheap, dict-merge friendly). The one field that comes from the LLM — the intent classification — is a Pydantic model, so the untrusted JSON is validated at the boundary before it ever lands in state.

In [ ]:
from typing import Literal

from pydantic import Field


# Pydantic model = the validated boundary for whatever the LLM returns.
class IntentResult(BaseModel):
    intent: Literal["search", "chitchat", "out_of_scope"]
    confidence: float = Field(ge=0.0, le=1.0)


# TypedDict = cheap internal graph state. One field holds the Pydantic value.
class RAGState(TypedDict):
    query: str
    intent: IntentResult
    documents: list[str]


# Pretend this raw JSON string came back from the LLM.
raw_llm_output = '{"intent": "search", "confidence": "0.92"}'

# Validate + coerce at the boundary ("0.92" -> 0.92).
parsed_intent = IntentResult.model_validate_json(raw_llm_output)

# Now place the trusted value into the plain-dict state.
state: RAGState = {
    "query": "how does langgraph handle interrupts?",
    "intent": parsed_intent,
    "documents": [],
}

print(type(state).__name__, "->", state)
print("intent:", state["intent"].intent, "| confidence:", state["intent"].confidence)

In [ ]:
# If the LLM hallucinates an invalid intent, it's rejected at the boundary
# instead of silently corrupting the state.
bad_llm_output = '{"intent": "delete_everything", "confidence": 0.99}'

try:
    IntentResult.model_validate_json(bad_llm_output)
except ValidationError as e:
    print("Rejected invalid LLM output:\n", e)